[<a href="https://colab.research.google.com/github/icsl-aist/hsr-genesis/blob/main/examples/tutorials/9_grasp_learning_cmaes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HSR Grasp Parameter Learning with CMA-ES / CMA-ESによる把持パラメータ学習

**目的 / Objective:**

- **CMA-ES** (Covariance Matrix Adaptation Evolution Strategy) を用いて、IK把持パイプラインの把持パラメータを最適化する方法を学ぶ / Learn to optimize grasp parameters of the IK pick pipeline using **CMA-ES**.
- YCB物体（**apple**）に対して、最適な把持高さ・グリッパ力・保持時間を進化戦略で探索する / Search for optimal grasp height, gripper effort, and hold time for a YCB object (**apple**) using an evolution strategy.
- EvoTorchのGPU加速CMA-ESとGenesis並列シミュレーションを組み合わせた高速パラメータ最適化を体験する / Experience fast parameter optimization combining EvoTorch's GPU-accelerated CMA-ES with Genesis parallel simulation.

> **Note:** Run this on a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU* or better). This notebook focuses on a **single object (apple)** — like the PPO tutorial — so CMA-ES optimizes only 4 parameters and finishes in ~2 min on a T4. To optimize all 7 objects (28D), use `train_grasp_cmaes.py` on a local GPU.

## 概要 / Overview

前のチュートリアルでは、IK把持パイプライン (アプローチ → 降下 → 把持 → 持ち上げ) を固定パラメータで実行しました。このノートブックでは、**把持パラメータを学習可能**にし、CMA-ESで最適化します。

In the previous tutorial, the IK grasp pipeline ran with fixed parameters. This notebook makes the **grasp parameters learnable** and optimizes them with CMA-ES.

### 最適化するパラメータ / Parameters to optimize

| Parameter | Range | Description |
|-----------|-------|-------------|
| `pre_grasp_height` | 0.05–0.30 m | Hover height above object before descending / 降下前のホバー高さ |
| `grasp_offset_z` | -0.02–0.08 m | Final grasp height relative to object center / 最終把持高さ |
| `gripper_effort` | 1.0–8.0 N | Force applied when closing gripper / グリッパ閉鎖時の力 |
| `grasp_hold_steps` | 100–500 steps | Steps to hold while gripper closes / 把持保持ステップ数 |

PPOチュートリアルと同様に、**apple (ycb_013_apple)** に焦点を当てます。apple はベースライン成功率が約33%と低く、改善の余地が大きいため、CMA-ESの効果が分かりやすいです。4パラメータの4次元探索空間を最適化します。

Like the PPO tutorial, we focus on **apple (ycb_013_apple)**. Apple has a low baseline success rate (~33%), leaving plenty of room for improvement — making CMA-ES effects clearly visible. We optimize a 4-dimensional search space (4 parameters).

## 1. Setup / セットアップ

依存パッケージのインストール、リポジトリのクローン、GPU レンダリングの設定を行います。
Install dependencies, clone the repo, and configure GPU rendering.

In [ ]:
import importlib, urllib.request

# Verify GPU runtime (Colab: Runtime → Change runtime type → T4 GPU)
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected. In Colab: Runtime → Change runtime type → T4 GPU. '
        'This notebook requires a GPU to run Genesis parallel simulation.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}')

# Fetch standalone bootstrap from GitHub (zero hsr_genesis imports).
exec(urllib.request.urlopen(
    "https://raw.githubusercontent.com/icsl-aist/hsr-genesis/main/colab_setup.py"
).read())

# One-call setup: installs deps, clones repo (with submodules),
# configures EGL for headless GPU rendering.
setup_colab()

# Install EvoTorch for CMA-ES
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'evotorch', '-q'], check=True)
print('EvoTorch installed.')

In [ ]:
import sys, pathlib

# Add repo to path (same as other tutorials)
REPO_DIR = pathlib.Path('/content/hsr-genesis')
SRC_DIR = str(REPO_DIR / 'src')
EXAMPLES_DIR = str(REPO_DIR / 'examples' / 'rl')
for p in [SRC_DIR, EXAMPLES_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import genesis as gs
if not getattr(gs, '_initialized', False):
    gs.init(backend=gs.gpu)
else:
    print('Genesis already initialized.')

import torch
import numpy as np
print(f'Genesis device: {gs.device}')

## 2. Grasp Parameters / 把持パラメータ

把持パラメータの定義と範囲を確認します。このノートブックではapple1物体に焦点を当てるため、**4次元ベクトル** = 4パラメータ × 1物体を最適化します（全7物体の28次元は `train_grasp_cmaes.py` で対応）。
Inspect the grasp parameter definitions and bounds. This notebook focuses on apple, so we optimize a **4D vector** = 4 params × 1 object (the full 7-object 28D case is handled by `train_grasp_cmaes.py`).

In [ ]:
from grasp_params import (
    PARAM_NAMES, OBJECT_NAMES, PARAM_BOUNDS,
    SOLUTION_LENGTH, N_PARAMS, N_OBJECTS,
    PARAM_DEFAULTS, default_params,
)

# Focus on a single object (same as PPO tutorial) for fast Colab training.
# apple has ~33% baseline — more room for CMA-ES to improve than foam_brick (80%).
OBJECT = 'ycb_013_apple'
OBJECT_IDX = OBJECT_NAMES.index(OBJECT)

print(f'Parameters: {PARAM_NAMES}')
print(f'All objects: {OBJECT_NAMES}')
print(f'Target object: {OBJECT} (index {OBJECT_IDX})')
print(f'Solution length: {N_PARAMS} (4 params x 1 object)')
print(f'  (full 7-object search = {SOLUTION_LENGTH}D, via train_grasp_cmaes.py)')
print()
print('Parameter bounds:')
for i, name in enumerate(PARAM_NAMES):
    lo, hi = PARAM_BOUNDS[i]
    print(f'  {name:25s} [{lo:.2f}, {hi:.2f}]')
print()
print(f'Default params: {PARAM_DEFAULTS.tolist()}')

## 3. Baseline Evaluation / ベースライン評価

デフォルトパラメータで代表的な3物体の把持成功率を測定します。apple（最適化対象）に加え、foam_brick（高いベースライン）とtennis_ball（難しい球形体）を比較用に含めます。
Measure pick success rates with default parameters on 3 representative objects: apple (our optimization target), plus foam_brick (high baseline) and tennis_ball (hard round object) for context.

### カーネルウォームアップ / Kernel Warmup

Colabの初回実行時、Quadrants（GenesisのGPUバックエンド）がNVRTCでGPUカーネルをJITコンパイルするため、最初の`scene.build()`とパイプライン実行に数分かかります。以下のウォームアップセルで小さなシーンをビルドし、全パイプライン（IK、軌道制御、グリッパ力、状態読み出し）を実行することで、すべてのカーネルを事前コンパイルします。Quadrantsのオフラインキャッシュはデフォルトで有効で、ディスクに自動保存されます。

On first Colab run, Quadrants JIT-compiles GPU kernels via NVRTC, making the first `scene.build()` and pipeline run take several minutes. The warmup below builds a tiny scene and runs the full pick pipeline to pre-compile all kernels (IK, trajectory control, gripper force, state readback). Quadrants offline cache is enabled by default and writes PTX to disk automatically.

In [ ]:
# Warmup: build a tiny scene and run a mini pick pipeline to pre-compile
# quadrants GPU kernels (NVRTC).
#
# On Colab, the quadrants PTX cache starts empty. The first call to each
# unique GPU kernel triggers NVRTC JIT compilation, which takes seconds
# per kernel with no log output. scene.step() alone only compiles physics
# step kernels — IK, trajectory control, gripper force, and state readback
# each have their own kernels that are only compiled when first called.
# We must exercise the full pick pipeline to compile everything.
#
# Quadrants offline cache is enabled by default and writes PTX to disk
# automatically — no explicit flush needed.
import time as _time
from ycb_pick_ik_parallel import HSRPickEnv

print('Warming up quadrants GPU kernel cache (one-time cost on Colab)...')
print(f'  Building 8-env scene ({OBJECT}) + running mini pipeline...')
_t0 = _time.time()

_warmup_env = HSRPickEnv(
    n_envs=8,
    object_name=OBJECT,
    show_viewer=False,
    seed=0,
    disable_visualizer=True,
)
_warmup_env.grasp_params = None  # use defaults
_warmup_env.run_pick_pipeline(settle_steps=2)
del _warmup_env
torch.cuda.empty_cache()

print(f'Warmup done ({_time.time() - _t0:.1f}s). '
      'Kernels compiled & cached to disk. Subsequent builds will be faster.')

In [ ]:
from ycb_pick_ik_parallel import HSRPickEnv

N_ENVS = 64  # enough for stable success rate estimates; 256 is 2x slower with little gain
SETTLE_STEPS = 30

default_matrix = default_params().to(gs.device)  # (7, 4)

# Evaluate 3 representative objects for context (instead of all 7, to save time):
#   - apple:        our optimization target (~33% baseline)
#   - foam_brick:   high baseline (~80%) — easy object for reference
#   - tennis_ball:  hard round object — shows top-down grasp limits
BASELINE_OBJECTS = ['ycb_061_foam_brick', OBJECT, 'ycb_056_tennis_ball']

baseline_rates = {}
for obj_name in BASELINE_OBJECTS:
    obj_idx = OBJECT_NAMES.index(obj_name)
    env = HSRPickEnv(
        n_envs=N_ENVS,
        object_name=obj_name,
        show_viewer=False,
        seed=42,
        disable_visualizer=True,
    )
    env.grasp_params = default_matrix[obj_idx:obj_idx+1].to(gs.device).expand(N_ENVS, -1)
    env.run_pick_pipeline(settle_steps=SETTLE_STEPS)
    summary = env.get_eval_summary()
    baseline_rates[obj_name] = summary['success_rate']
    marker = ' <-- target' if obj_name == OBJECT else ''
    print(f'  {obj_name:25s} {summary["success_rate"]:.2%}  (n={summary["n_envs"]}){marker}', flush=True)
    del env
    torch.cuda.empty_cache()

baseline_apple = baseline_rates[OBJECT]
print(f'\nBaseline rates (3 representative objects):')
for o in BASELINE_OBJECTS:
    marker = ' <-- target' if o == OBJECT else ''
    print(f'  {o:25s} {baseline_rates[o]:.2%}{marker}')

## 4. CMA-ES Optimizer / CMA-ES最適化器

EvoTorchのCMA-ESを使用します。各世代で候補パラメータをサンプリングし、**apple1物体**で評価して適応度を計算します。4次元探索空間なので1世代あたりの評価は1回のパイプライン実行のみ（全7物体版の1/7の時間）。

We use EvoTorch's CMA-ES. Each generation samples candidate parameter vectors, evaluates them on **apple only**, and computes fitness. The 4D search space means each generation runs just one pick pipeline batch — 7x faster than the full 7-object version.

### アルゴリズムの仕組み / How it works

1. **サンプリング / Sampling**: CMA-ESが多変量正規分布から候補を生成 / CMA-ES samples candidates from a multivariate Gaussian
2. **評価 / Evaluation**: 各候補をappleでIK把持パイプライン実行 / Run IK pick pipeline for each candidate on apple
3. **適応度 / Fitness**: appleの成功率 / Success rate on apple
4. **更新 / Update**: 適応度に基づいて分布の平均・共分散・ステップサイズを更新 / Update distribution mean, covariance, step size based on fitness

![CMA-ES concept](https://upload.wikimedia.org/wikipedia/commons/thumb/d/d8/Concept_of_directional_optimization_in_CMA-ES_algorithm.png/500px-Concept_of_directional_optimization_in_CMA-ES_algorithm.png)

In [ ]:
from evotorch import Problem
from evotorch.algorithms import CMAES
from ycb_pick_ik_parallel import HSRPickEnv
import time as _time

class GraspProblem(Problem):
    """EvoTorch Problem: evaluate grasp params via IK pick simulation.

    Single-object mode: optimizes 4 params (N_PARAMS) for one object.
    This is 7x faster per generation than the full 7-object 28D version
    (train_grasp_cmaes.py), making it feasible on Colab T4 in ~2 min.
    """

    def __init__(self, *, popsize, object_name, settle_steps, seed=0):
        super().__init__(
            objective_sense='max',
            solution_length=N_PARAMS,  # 4D: single object
            initial_bounds=(0.0, 1.0),  # normalized search space
            device='cpu',
        )
        self.popsize = popsize
        self.object_name = object_name
        self.settle_steps = settle_steps
        self.seed = seed
        self._env: HSRPickEnv | None = None

    def _get_env(self) -> HSRPickEnv:
        if self._env is None:
            self._env = HSRPickEnv(
                n_envs=self.popsize,
                object_name=self.object_name,
                show_viewer=False,
                seed=self.seed,
                disable_visualizer=True,
            )
        return self._env

    def _evaluate_batch(self, solutions):
        n = solutions.values.shape[0]
        raw = solutions.values.clone()  # (n, 4) in [0,1]

        # Denormalize [0,1] -> actual param ranges (4 params, single object)
        lo = PARAM_BOUNDS[:, 0].to(raw.device)
        hi = PARAM_BOUNDS[:, 1].to(raw.device)
        scaled = raw.clamp(0.0, 1.0) * (hi - lo) + lo  # (n, 4)
        scaled[..., 3] = torch.round(scaled[..., 3])  # round hold steps to int
        clipped = scaled.to(device=gs.device, dtype=gs.tc_float)

        t_obj = _time.time()
        env = self._get_env()
        env.grasp_params = clipped
        result = env.run_pick_pipeline(settle_steps=self.settle_steps)
        fitness = result['success_per_env'].to(torch.float32)  # (n,)

        solutions.set_evals(fitness)

        rate = float(result['success_rate'])
        print(f'    [{self.object_name}] {rate:.2%}  '
              f'({_time.time() - t_obj:.1f}s)', flush=True)

        best_idx = int(fitness.argmax())
        print(f'  [eval] best={float(fitness[best_idx]):.3f} '
              f'mean={float(fitness.mean()):.3f}')

print('GraspProblem defined (single-object, 4D).')

### CMA-ES vs 遺伝的アルゴリズム / CMA-ES vs Genetic Algorithm

なぜCMA-ESがこの問題に適しているのか、遺伝的アルゴリズム (GA) と比較して理解しましょう。

Let's understand why CMA-ES is well-suited for this problem by contrasting it with a Genetic Algorithm (GA).

| Aspect / 側面 | Genetic Algorithm (GA) | CMA-ES |
|---|---|---|
| **表現 / Representation** | 個体を染色体（離散/連続）として符号化 / Encode individuals as chromosomes (discrete/continuous) | 連続ベクトル（多変量ガウス分布） / Continuous vector (multivariate Gaussian) |
| **仕組み / Mechanism** | 選択→交差→突然変異を個体に適用 / Selection → Crossover → Mutation on individuals | ガウス分布 N(m, C, σ²) からサンプリング→適応度で平均・共分散・ステップサイズを更新 / Sample from Gaussian, update mean, covariance, step size |
| **適応 / Adaptation** | 突然変異率は固定（またはスケジュール） / Mutation rate is fixed (or scheduled) | ステップサイズ σ と共分散行列 C が**自動適応** / Step size σ and covariance matrix C **automatically adapt** |
| **活用情報 / Info used** | 個体の優劣（適応度ランキング）のみ / Only individual superiority (fitness ranking) | ランキング**＋分布構造**（変数間相関） / Ranking **+ distribution structure** (inter-variable correlations) |
| **勾配 / Gradient** | 不要 / Not required | 不要（ブラックボックス最適化）/ Not required (black-box optimization) |
| **適した空間 / Best search space** | 離散/組合せ最適化（ビット列、順列） / Discrete/combinatorial (bit strings, permutations) | 連続値（実数ベクトル） / Continuous (real-valued vector) |
| **集団 / Population** | 固定サイズ、トーナメント/ランキング選択 / Fixed size, tournament/ranking selection | 適応的サンプリング、集団サイズ可変 / Adaptive sampling, changeable popsize |
| **収束 / Convergence** | 突然変異率固定だと最適解付近で減速 / Slows near optimum when mutation rate is fixed | 最適解への幾何収束（理論保証）/ Geometric convergence to optimum (proven) |

### なぜこの問題にCMA-ESか / Why CMA-ES for this problem

- **28次元連続サーチ空間**: 4パラメータ × 7物体 = 実数ベクトル → GAの離散交差より連続ガウスサンプリングが自然 / 28D continuous space: real-valued vector → continuous Gaussian sampling more natural than GA's discrete crossover
- **変数間相関のキャプチャ**: 把持高さが低い↔グリッパ力を高く必要、などの相関を共分散行列が学習 / Covariance captures correlations (e.g. low grasp height ↔ higher effort needed)
- **疎な報酬**: 成功/失敗の二値報酬 → 勾配ベース法が困難だが、ランキングベースのCMA-ESは機能 / Sparse binary reward → gradient methods struggle, but ranking-based CMA-ES works
- **軸非整列の相関**: GAの交差は軸に沿った組み合わせしか生成しないが、CMA-ESの共分散行列は任意方向の相関を表現可能 / GA crossover only combines along axes; CMA-ES covariance can represent correlations in any direction

### 1次元デモ: GA vs CMA-ES / 1D Demo: GA vs CMA-ES

1次元の多峰性地形で両アルゴリズムの挙動を可視化します。GAは集団が良い領域に集中する様子、CMA-ESはガウス分布が最適解に向かって収縮する様子が観察できます。

Visualize both algorithms on a 1D multimodal landscape. Observe GA's population clustering vs CMA-ES's Gaussian distribution shrinking toward the optimum.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 1D landscape: multimodal function
def landscape(x):
    return np.sin(3 * x) * np.exp(-0.1 * x**2)

x_grid = np.linspace(-5, 5, 300)
y_grid = landscape(x_grid)

# --- GA simulation ---
rng_ga = np.random.default_rng(0)
pop_ga = rng_ga.uniform(-5, 5, 30)
ga_history = [pop_ga.copy()]
for _ in range(20):
    fitness = landscape(pop_ga)
    idx = np.argsort(fitness)[::-1][:15]
    parents = pop_ga[idx]
    children = rng_ga.choice(parents, 15) + rng_ga.normal(0, 0.5, 15)
    pop_ga = np.concatenate([parents, children])
    ga_history.append(pop_ga.copy())

# --- CMA-ES simulation (simplified 1D) ---
rng_cma = np.random.default_rng(0)
mean = 0.0
sigma = 2.0
cma_means = []
cma_samples_history = []
for _ in range(20):
    samples = rng_cma.normal(mean, sigma, 30)
    fitness = landscape(samples)
    idx = np.argsort(fitness)[::-1]
    mu = 15
    weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    weights /= weights.sum()
    mean = np.sum(weights * samples[idx[:mu]])
    success_rate = np.mean(fitness > landscape(np.full(30, mean)))
    sigma *= np.exp(0.1 * (success_rate - 0.2))
    sigma = max(sigma, 0.01)
    cma_means.append(mean)
    cma_samples_history.append(samples.copy())
cma_means.append(mean)
cma_samples_history.append(rng_cma.normal(mean, sigma, 30))

# --- Animation ---
n_frames = len(ga_history)
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

for ax in axes:
    ax.plot(x_grid, y_grid, 'k-', alpha=0.3, linewidth=2)
    ax.set_xlim(-5, 5)
    ax.set_xlabel('x')
    ax.set_ylabel('fitness')

axes[0].set_title('Genetic Algorithm: population per generation')
axes[1].set_title('CMA-ES: Gaussian samples per generation')

ga_scat = axes[0].scatter([], [], c='tab:blue', alpha=0.6, s=30, zorder=3)
ga_text = axes[0].text(0.02, 0.95, '', transform=axes[0].transAxes, va='top')

cma_scat = axes[1].scatter([], [], c='tab:orange', alpha=0.6, s=30, zorder=3)
cma_mean_line = axes[1].axvline(0, color='red', linestyle='--', alpha=0.5, linewidth=1)
_sigma_fill = {'artist': axes[1].fill_between(
    x_grid, -2, 2, where=np.zeros(len(x_grid), dtype=bool),
    color='red', alpha=0.1,
)}
cma_text = axes[1].text(0.02, 0.95, '', transform=axes[1].transAxes, va='top')

def animate(i):
    # GA
    ga_pop = ga_history[i]
    ga_fit = landscape(ga_pop)
    ga_scat.set_offsets(np.column_stack([ga_pop, ga_fit]))
    ga_text.set_text(f'Gen {i}')

    # CMA-ES
    cma_pop = cma_samples_history[i]
    cma_fit = landscape(cma_pop)
    cma_scat.set_offsets(np.column_stack([cma_pop, cma_fit]))
    m = cma_means[i]
    cma_mean_line.set_xdata([m, m])
    _sigma_fill['artist'].remove()
    _sigma_fill['artist'] = axes[1].fill_between(
        x_grid, -2, 2, where=np.abs(x_grid - m) < sigma,
        color='red', alpha=0.1,
    )
    cma_text.set_text(f'Gen {i}  mean={m:.2f}  σ={sigma:.2f}')

    return ga_scat, cma_scat

anim = FuncAnimation(fig, animate, frames=n_frames, interval=400, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 5. Training / 学習

CMA-ESでappleの把持パラメータを最適化します。4次元探索空間 + 単一物体評価のため、Colab T4でも数分で完了します。

Optimize apple's grasp parameters with CMA-ES. The 4D search space + single-object evaluation means it finishes in a few minutes even on Colab T4.

> **Tip:** `GENERATIONS` を増やす（例: 30）とより良い結果が得られます / Increase `GENERATIONS` (e.g. to 30) for better results.

In [ ]:
import time, json

POPSIZE = 64       # 64-way parallel; 256 is 2x slower with little quality gain
GENERATIONS = 15   # 4D converges fast; increase to 30 for better results
SETTLE = 30
OUTPUT_DIR = pathlib.Path('/content/grasp_cmaes_results')
OUTPUT_DIR.mkdir(exist_ok=True)

problem = GraspProblem(
    popsize=POPSIZE,
    object_name=OBJECT,
    settle_steps=SETTLE,
    seed=0,
)

# Initialize CMA-ES at normalized default params (4D, single object)
lo = PARAM_BOUNDS[:, 0]  # (4,)
hi = PARAM_BOUNDS[:, 1]  # (4,)
defaults_norm = (PARAM_DEFAULTS - lo) / (hi - lo)  # (4,)

cmaes = CMAES(
    problem=problem,
    stdev_init=0.2,
    popsize=POPSIZE,
    center_init=defaults_norm,
)

best_fitness = -1.0
best_params = None
history = []

for gen in range(GENERATIONS):
    t0 = time.time()
    cmaes.step()
    pop = cmaes.population
    evals = pop.evals
    best_f = float(evals.max())
    mean_f = float(evals.mean())
    dt = time.time() - t0

    if best_f > best_fitness:
        best_fitness = best_f
        best_idx = int(evals.argmax())
        best_raw = pop.values[best_idx].clone().cpu()
        scaled = best_raw.clamp(0, 1) * (hi - lo) + lo  # (4,)
        scaled[3] = torch.round(scaled[3])
        best_params = {
            'pre_grasp_height': float(scaled[0]),
            'grasp_offset_z': float(scaled[1]),
            'gripper_effort': float(scaled[2]),
            'grasp_hold_steps': int(scaled[3]),
        }

    history.append({'gen': gen, 'best': best_f, 'mean': mean_f})
    print(f'[gen {gen:>3d}] best={best_f:.3f} mean={mean_f:.3f} '
          f'({dt:.1f}s) overall_best={best_fitness:.3f}')

    # Checkpoint
    with open(OUTPUT_DIR / 'grasp_cmaes_best.json', 'w') as f:
        json.dump({
            'object': OBJECT,
            'generation': gen,
            'fitness': best_fitness,
            'params': best_params,
        }, f, indent=2)

print(f'\nTraining complete. Best fitness: {best_fitness:.3f}')
print(f'Baseline apple: {baseline_apple:.3f}')
print(f'Improvement:    {best_fitness - baseline_apple:+.3f}')

## 6. Training Curve / 学習曲線

世代ごとの適応度推移を可視化します。
Visualize fitness progression across generations.

In [ ]:
import matplotlib.pyplot as plt

gens = [h['gen'] for h in history]
bests = [h['best'] for h in history]
means = [h['mean'] for h in history]

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(gens, bests, 'o-', label='Best fitness', color='tab:blue')
ax.plot(gens, means, 's--', label='Mean fitness', color='tab:orange', alpha=0.7)
ax.axhline(y=baseline_apple, color='tab:red', linestyle=':', label=f'Baseline apple ({baseline_apple:.2f})')
ax.set_xlabel('Generation')
ax.set_ylabel('Success rate (apple)')
ax.set_title(f'CMA-ES Grasp Parameter Optimization — {OBJECT}')
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Learned Parameters / 学習済みパラメータ

appleの最適化されたパラメータをデフォルトと比較します。
Compare apple's optimized parameters against defaults.

In [ ]:
print(f'Learned grasp parameters for {OBJECT}:')
print(f'{"Parameter":25s} {"Default":>10s} {"Learned":>10s} {"Delta":>10s}')
print('-' * 58)
defaults_list = PARAM_DEFAULTS.tolist()
for i, name in enumerate(PARAM_NAMES):
    dval = defaults_list[i]
    lval = best_params[name]
    if name == 'grasp_hold_steps':
        print(f'{name:25s} {dval:10d} {lval:10d} {lval - dval:+10d}')
    else:
        print(f'{name:25s} {dval:10.4f} {lval:10.4f} {lval - dval:+10.4f}')

print(f'\nDefaults: {PARAM_DEFAULTS.tolist()}')
print(f'\nInsights:')
effort_diff = best_params['gripper_effort'] - float(PARAM_DEFAULTS[2])
height_diff = best_params['pre_grasp_height'] - float(PARAM_DEFAULTS[0])
notes = []
if effort_diff > 1.0:
    notes.append(f'+{effort_diff:.1f}N effort')
elif effort_diff < -1.0:
    notes.append(f'{effort_diff:.1f}N effort')
if height_diff > 0.05:
    notes.append(f'+{height_diff:.2f}m height')
elif height_diff < -0.05:
    notes.append(f'{height_diff:.2f}m height')
if notes:
    print(f'  {OBJECT}: {" ".join(notes)}')
else:
    print(f'  {OBJECT}: params close to defaults')

## 8. Final Evaluation / 最終評価

学習済みパラメータでappleを再評価し、ベースラインと比較します。
Re-evaluate apple with learned parameters and compare against baseline.

In [ ]:
from ycb_pick_ik_parallel import HSRPickEnv

# Build a (4,) tensor from the learned single-object params dict
learned_vec = torch.tensor([
    best_params['pre_grasp_height'],
    best_params['grasp_offset_z'],
    best_params['gripper_effort'],
    best_params['grasp_hold_steps'],
], dtype=torch.float32, device=gs.device).expand(N_ENVS, -1)

env = HSRPickEnv(
    n_envs=N_ENVS,
    object_name=OBJECT,
    show_viewer=False,
    seed=42,
    disable_visualizer=True,
)
env.grasp_params = learned_vec
env.run_pick_pipeline(settle_steps=SETTLE_STEPS)
summary = env.get_eval_summary()
learned_apple = summary['success_rate']
del env
torch.cuda.empty_cache()

print(f'{OBJECT:25s} {"Baseline":>10s} {"Learned":>10s} {"Delta":>8s}')
print('-' * 55)
delta = learned_apple - baseline_apple
print(f'{OBJECT:25s} {baseline_apple:10.2%} {learned_apple:10.2%} {delta:+8.2%}')
print(f'\nBaseline apple: {baseline_apple:.2%}  ->  Learned: {learned_apple:.2%}  ({delta:+.2%})')

## 9. Per-Object Comparison Bar Chart / 物体別比較

3物体のベースライン成功率と、appleの学習済み成功率を棒グラフで比較します。
Compare baseline success rates across the 3 representative objects, with apple's learned rate highlighted.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

short_names = [o.replace('ycb_', '') for o in BASELINE_OBJECTS]
x = np.arange(len(BASELINE_OBJECTS))
width = 0.35

# Baseline bars for the 3 representative objects
baseline_vals = [baseline_rates[o] for o in BASELINE_OBJECTS]
bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline (defaults)', color='tab:red', alpha=0.7)

# Learned bar only for the target object (apple); others get 0 (not trained)
learned_vals = [learned_apple if o == OBJECT else 0 for o in BASELINE_OBJECTS]
target_idx = BASELINE_OBJECTS.index(OBJECT)
bars2 = ax.bar(x + width/2, learned_vals, width, label=f'Learned (CMA-ES, {OBJECT.split("_")[-1]})', color='tab:blue', alpha=0.7)

ax.set_ylabel('Success rate')
ax.set_title(f'Grasp Success: Baseline (3 objects) vs CMA-ES Learned ({OBJECT})')
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=30, ha='right')
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h:.0%}',
            ha='center', va='bottom', fontsize=8)
# Only label the learned bar for the target object
bar = bars2[target_idx]
h = bar.get_height()
ax.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h:.0%}',
        ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Save & Load / 保存と読み込み

学習済みパラメータを保存・読み込みする方法を示します。
Show how to save and load learned parameters.

In [ ]:
# Save is already done during training, but show the format:
checkpoint_path = OUTPUT_DIR / 'grasp_cmaes_best.json'
print(f'Checkpoint saved at: {checkpoint_path}')
print()

with open(checkpoint_path) as f:
    ckpt = json.load(f)
print(f'Object:    {ckpt["object"]}')
print(f'Generation: {ckpt["generation"]}')
print(f'Fitness:    {ckpt["fitness"]:.3f}')
print(f'Params:     {ckpt["params"]}')
print()
print('To load in your own script:')
print('''
  import json
  with open("grasp_cmaes_best.json") as f:
      ckpt = json.load(f)
  params = ckpt["params"]  # dict: {pre_grasp_height, grasp_offset_z, gripper_effort, grasp_hold_steps}
  # Build a (4,) tensor for HSRPickEnv:
  import torch
  vec = torch.tensor([params["pre_grasp_height"], params["grasp_offset_z"],
                      params["gripper_effort"], params["grasp_hold_steps"]])
  env.grasp_params = vec.to(gs.device).expand(n_envs, -1)
''')
print('To optimize all 7 objects (28D), use train_grasp_cmaes.py on a local GPU:')
print('  PYTHONPATH=src .venv/bin/python examples/rl/train_grasp_cmaes.py \\\n'
      '    --popsize 256 --generations 50')

## まとめ / Summary

CMA-ESによる把持パラメータ学習の流れを振り返ります / Review of CMA-ES grasp parameter learning:

| Step | Component | Description |
|------|-----------|-------------|
| Parameters | `grasp_params.py` | 4 params × 1 object (apple) = 4D search space / 4次元探索空間 |
| Environment | `HSRPickEnv` | Parameterized IK pick pipeline / パラメータ化IK把持パイプライン |
| Optimizer | `GraspProblem` + `CMAES` | EvoTorch GPU CMA-ES / EvoTorch GPU CMA-ES |
| Training | `train_grasp_cmaes.py` | Full 7-object 28D training script (local GPU) / 全7物体28D学習スクリプト（ローカルGPU） |
| Evaluation | `eval_grasp_params.py` | Load checkpoint, eval on all objects / チェックポイント読み込み・全物体評価 |

### 重要なポイント / Key takeaways

- **CMA-ESはブラックボックス最適化**: 勾配不要、報酬が離散的（成功/失敗）でも機能する / CMA-ES is black-box optimization: no gradients needed, works with sparse binary rewards
- **単一物体フォーカスで高速化**: PPOチュートリアルと同様にappleに焦点を当て、4D探索空間・1物体評価でColab T4でも数分で完了 / Single-object focus (like PPO tutorial): 4D search + 1-object eval finishes in minutes on Colab T4
- **並列シミュレーションが鍵**: 64環境の並列IK把持で1世代を数秒で完了 / Parallel simulation is key: 64-env parallel IK pick completes a generation in seconds
- **全7物体最適化も可能**: `train_grasp_cmaes.py` で28D（4パラメータ×7物体）をローカルGPUで最適化 / Full 7-object optimization available: `train_grasp_cmaes.py` optimizes 28D on a local GPU

### 次のステップ / Next steps

- **より大規模な訓練 / Larger training**: `GENERATIONS` を30以上に増やしてより良い結果を得る / Increase `GENERATIONS` to 30+ for better results
- **別の物体で試す / Try other objects**: `OBJECT` を変更して別の物体を最適化 / Change `OBJECT` to optimize a different object
- **全7物体の最適化 / All 7 objects**: `train_grasp_cmaes.py` で28D最適化をローカルGPUで実行 / Run `train_grasp_cmaes.py` for 28D optimization on a local GPU
- **PPO残差ポリシー / PPO residual policy**: 次のチュートリアルでCMA-ESパラメータの上にNN残差補正を学習 / Next tutorial: learn NN residual corrections on top of CMA-ES params
- **把持戦略の選択 / Grasp strategy selection**: トップダウン・サイド・ピンチの戦略選択を学習 / Learn to select grasp strategies (top, side, pinch)